Imports & Dataset Loading (Vector Store Construction)

In [1]:
import pandas as pd

from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import numpy as np
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from new_config_propmt import Config
from evaluate import evaluate_model
import importlib
import new_config_propmt as config_prompt
importlib.reload(config_prompt)
import evaluate
importlib.reload(evaluate)


/home/anuj97og/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'evaluate' from '/home/anuj97og/FakeReview/fake_reviews_prediction/Prompting/Old_files (Need to check)/evaluate.py'>

In [2]:
import os
print(os.environ.get("OLLAMA_HOST")) 

os.environ["OLLAMA_HOST"] = "http://localhost:8000"

None


In [3]:
print(os.environ.get("OLLAMA_HOST")) 

http://localhost:8000


In [33]:
pip install langchain faiss-cpu sentence-transformers pandas


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/usr/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [8]:
import langchain_community
print(langchain_community.__version__)


0.3.31


In [90]:
from sklearn.model_selection import train_test_split
csv_path = r"/home/anuj97og/FakeReview/fake_reviews_prediction/Datasets/Hotel_HumanReal_VS_CG.csv"
df = pd.read_csv(csv_path)
df.rename(columns={"Binaery_label": "label"}, inplace=True)

df_set1, df_set2 = train_test_split(
    df,
    test_size=0.5,
    random_state=42,
    stratify=df["label"]
)

In [91]:
print(len(df))
print(len(df_set1))
print(len(df_set2))

1592
796
796


Convert Rows into LangChain Documents

In [92]:
documents = []

for _, row in df_set1.iterrows():
    documents.append(
        Document(
            page_content=row["text"],
            metadata={"label": row["label"]}
        )
    )


Create Embeddings & Vector Store

In [93]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


In [94]:
vectorstore = FAISS.from_documents(documents, embeddings)


In [95]:
vectorstore.save_local("hotel_review_vectorstore")


Retrieval Test (Sanity Check)

In [96]:
vectorstore = FAISS.load_local(
    "hotel_review_vectorstore",
    embeddings,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


In [97]:
query = "Absolutely perfect hotel, best stay ever!!!"

docs = retriever.invoke(query)

for d in docs:
    print("TEXT:", d.page_content)
    print("LABEL:", d.metadata["label"])
    print("-" * 40)


TEXT: Great hotel in a wonderful location. I stayed in this hotel last weekend for one night before coming home and I loved it. I couldn't recommend it highly enough. Evidently popular and busy, the staff were still helpful. The rooms were lovely and spacious and clean. Bathrooms were clean, bright and spacious also. I'd definitely recommend this hotel too, it's close to the magnificant mile. 

LABEL: real
----------------------------------------
TEXT: Excellent Hotel ! Rooms and service were great. A great value ! I have been to places that are not as good for nearly twice the price. Good location. Close to the L and short walk to rental car service. Restaurants in the area a bit of a disappointment, but the shopping was great. Area recommendations Great theatre, restaurants, museums and shopping.

LABEL: real
----------------------------------------
TEXT: After reading the reviews on this hotel, I decided to book my stay for July 4th weekend here and I am glad I did, this hotel is ab

RAG Prompt Definition

In [98]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}   # number of retrieved examples
)


In [ ]:
from langchain_core.prompts import PromptTemplate

rag_prompt = PromptTemplate(
    input_variables=["examples", "review"],
    template="""
Role
You are an expert linguistic analyst specializing in classifying real and fake online reviews.
Task
Classify a given hotel review as either "real" or "fake".

Context
You will classify the review based on linguistic cues that distinguish genuine customer experiences from artificially generated or promotional content.

Classification Rules for internal use only do NOT write them in output:
- Fake reviews often contain generic or templated language.
- Fake reviews may exaggerate praise or criticism.
- Marketing-style or promotional wording is a strong indicator of fake.
- Real reviews usually include natural language and specific experiential details.

Review to Classify
"{review}"

CRITICAL OUTPUT REQUIREMENT — STRICTLY ENFORCED
- You must output ONLY ONE WORD.
- Do NOT write classification Rules and Context.
- Do NOT provide explanations, reasoning, or analysis.
- Any additional text will invalidate the output.

Answer with your classification.
Use one of these labels:
"real" or "fake"

Take help from below examples of hotel reviews with their labels similar to above review.
{examples}

Respond with ONLY one word: real or fake.
"""
)


Format Retrieved Examples

In [100]:
def format_examples(docs):
    formatted = []
    for d in docs:
        formatted.append(
            f'Review: "{d.page_content}"\nLabel: {d.metadata["label"]}'
        )
    return "\n\n".join(formatted)


Load LLM model via Ollama

In [122]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="qwen:32b")


Output Normalization

In [123]:
def normalize_prediction(response):
    response = response.lower()

    if "real" in response:
        return "real"
    elif "fake" in response:
        return "fake"
    else:
        return "UNKNOWN"


RAG Classification Function

In [124]:
def classify_review_rag(review_text):
    # Retrieve similar labeled reviews
    docs = retriever.invoke(review_text)

    # Format them as few-shot examples
    examples = format_examples(docs)

    # Build prompt
    prompt = rag_prompt.format(
        examples=examples,
        review=review_text
    )

    # Call Gemma
    response = llm.invoke(prompt)

    # Clean output
    #label = response.strip().upper()
    #label = label.split()[0].strip(".,!?:;")
    label = normalize_prediction(response)

    return label


In [125]:
def classify_review_rag_withoutnormalization(review_text):
    # Retrieve similar labeled reviews
    docs = retriever.invoke(review_text)

    # Format them as few-shot examples
    examples = format_examples(docs)

    # Build prompt
    prompt = rag_prompt.format(
        examples=examples,
        review=review_text
    )

    # Call llama
    response = llm.invoke(prompt)
    response = response.strip().lower()
    if " " in response:
        label = response.split()[0]
    label = response.strip('.,!?;:')
    

    return label


In [126]:
df_set2.head()


,label,Category,domain,text,is_synthetic,source
368,real,homewood,Hotel,I visited Chicago with my two teenage daughter...,0,TripAdvisor
1462,fake,sofitel,Hotel,"Sure, here's my review for Sofitel:\n\nLocated...",1,granite3.1-moe:latest
1470,fake,sofitel,Hotel,Sofitel Hotel offers an unparalleled urban ret...,1,granite3.1-moe:latest
1531,fake,knickerbocker,Hotel,Knickerbocker Hotel - Exceptional Location and...,1,granite3.1-moe:latest
0,real,james,Hotel,My husband and I were there for a conference a...,0,Web


In [127]:
y_true = []
y_pred = []

for idx, row in df_set2.iterrows():
    review = row["text"]
    true_label = row["label"]
    pred_label = classify_review_rag(review)
    y_true.append(true_label)
    y_pred.append(pred_label)

    print(f"[{idx}] True: {true_label} | Pred: {pred_label}")


[368] True: real | Pred: real
[1462] True: fake | Pred: real
[1470] True: fake | Pred: fake
[1531] True: fake | Pred: real
[0] True: real | Pred: real
[458] True: real | Pred: real
[38] True: real | Pred: real
[985] True: fake | Pred: real
[314] True: real | Pred: real
[1145] True: fake | Pred: fake
[1580] True: fake | Pred: real
[72] True: real | Pred: real
[256] True: real | Pred: real
[947] True: fake | Pred: fake
[1272] True: fake | Pred: real
[176] True: real | Pred: real
[1025] True: fake | Pred: fake
[945] True: fake | Pred: fake
[1126] True: fake | Pred: real
[523] True: real | Pred: real
[1135] True: fake | Pred: fake
[1501] True: fake | Pred: real
[918] True: fake | Pred: real
[958] True: fake | Pred: fake
[285] True: real | Pred: real
[809] True: fake | Pred: real
[1495] True: fake | Pred: real
[1263] True: fake | Pred: real
[133] True: real | Pred: real
[473] True: real | Pred: real
[281] True: real | Pred: real
[502] True: real | Pred: real
[566] True: real | Pred: real
[1

Evaluation Pipeline

In [128]:
accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, pos_label="fake")
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=["fake", "real"]
)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print("-----Confusion Matrix-----")
print(cm)

Accuracy: 0.6269
F1 Score: 0.4142
-----Confusion Matrix-----
[[105 293]
 [  4 394]]


In [28]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=["real", "fake"]
)

print(cm)


[[144  56]
 [ 38 162]]


In [24]:
test_review = "My husband and I stayed here for New Years eve weekend. This is an excellent hotel in an excellent location. Hotel is very tastefully decorated and warmly welcoming. The rooms had the best electronics I've ever seen. Loved that they had a soaking tub and a spacious shower stall. The hotel restaurant seemed to be empty everytime we walked by, so we avoided it too. Ate at Joe's across the street - excellent seafood. And you can shop til you drop without going outside! Nordstroms and four story mall is attached. Lots of good 'food on the run' places in there. Loved Chicago - loved the Conrad. I will go back!"

result = classify_review_rag(test_review)
print("Predicted label:", result)


Predicted label: TRUTHFUL


In [7]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu


Looking in indexes: https://download.pytorch.org/whl/cpu
   ---------------------------------------- 0.0/110.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/110.9 MB 15.9 MB/s eta 0:00:07
   ---------------------------------------- 0.7/110.9 MB 8.4 MB/s eta 0:00:14
    --------------------------------------- 1.6/110.9 MB 12.7 MB/s eta 0:00:09
    --------------------------------------- 2.1/110.9 MB 12.3 MB/s eta 0:00:09
    --------------------------------------- 2.7/110.9 MB 12.3 MB/s eta 0:00:09
   - -------------------------------------- 2.9/110.9 MB 12.5 MB/s eta 0:00:09
   - -------------------------------------- 3.7/110.9 MB 12.4 MB/s eta 0:00:09
   - -------------------------------------- 4.3/110.9 MB 12.4 MB/s eta 0:00:09
   - -------------------------------------- 4.4/110.9 MB 11.2 MB/s eta 0:00:10
   - -------------------------------------- 5.2/110.9 MB 11.8 MB/s eta 0:00:09
   - -------------------------------------- 5.4/110.9 MB 11.1 MB/s eta 0:00:10
   -

In [8]:
pip install sentence-transformers


   ---------------------------------------- 0.0/493.7 kB ? eta -:--:--
    --------------------------------------- 10.2/493.7 kB ? eta -:--:--
    --------------------------------------- 10.2/493.7 kB ? eta -:--:--
    --------------------------------------- 10.2/493.7 kB ? eta -:--:--
   --- ----------------------------------- 41.0/493.7 kB 281.8 kB/s eta 0:00:02
   --- ----------------------------------- 41.0/493.7 kB 281.8 kB/s eta 0:00:02
   --- ----------------------------------- 41.0/493.7 kB 281.8 kB/s eta 0:00:02
   ----------- -------------------------- 143.4/493.7 kB 500.5 kB/s eta 0:00:01
   ---------------------------------- ----- 419.8/493.7 kB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 493.7/493.7 kB 1.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())  # should be False (this is OK)


2.9.1+cpu
False


In [12]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded successfully")


Model loaded successfully


In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded successfully")


c:\Users\suraj\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model loaded successfully
